# Retail Sales & Customer Analytics

**Tools:** Python, SQL, Power BI

**Goal:** identify revenue/profit drivers and customer segments, then convert the findings into business recommendations.

## 1. Load data

If the file is not present, run `src/download_data.py` first.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = Path('../data/Sample-Superstore.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('Download the dataset first with: python ../src/download_data.py')

df = pd.read_csv(DATA_PATH)
df.head()


## 2. Data understanding and quality checks

In [ ]:
print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())
print('\nMissing values:')
print(df.isna().sum().sort_values(ascending=False).head(15))
print('\nDuplicate rows:', df.duplicated().sum())
df.info()


## 3. Standardize fields

The dataset may use slightly different column naming conventions depending on the mirror. This section normalizes common names.

In [ ]:
rename_map = {
    'Order ID':'Order_ID','Order Date':'Order_Date','Ship Date':'Ship_Date',
    'Customer ID':'Customer_ID','Customer Name':'Customer_Name',
    'Product ID':'Product_ID','Product Name':'Product_Name',
    'Sub-Category':'Sub_Category'
}
df = df.rename(columns={k:v for k,v in rename_map.items() if k in df.columns})

for c in ['Order_Date','Ship_Date']:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors='coerce')

for c in ['Sales','Quantity','Discount','Profit']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

df = df.drop_duplicates().copy()
df['Profit_Margin'] = np.where(df['Sales'] != 0, df['Profit']/df['Sales'], np.nan)
df['Order_Month'] = df['Order_Date'].dt.to_period('M').astype(str)
df['Order_Year'] = df['Order_Date'].dt.year
df.head()


## 4. Executive KPIs

In [ ]:
kpis = {
    'Total Sales': df['Sales'].sum(),
    'Total Profit': df['Profit'].sum(),
    'Orders': df['Order_ID'].nunique(),
    'Customers': df['Customer_ID'].nunique(),
    'Profit Margin': df['Profit'].sum()/df['Sales'].sum()
}
pd.Series(kpis)


## 5. Sales and profit trends

In [ ]:
monthly = df.groupby('Order_Month', as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum'))
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(monthly['Order_Month'], monthly['Sales'], label='Sales')
ax.plot(monthly['Order_Month'], monthly['Profit'], label='Profit')
ax.set_title('Monthly Sales and Profit')
ax.tick_params(axis='x', rotation=60)
ax.legend()
plt.tight_layout(); plt.show()


## 6. Category and sub-category performance

In [ ]:
category = df.groupby('Category', as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum'))
category['Profit_Margin'] = category['Profit']/category['Sales']
display(category.sort_values('Sales', ascending=False))

subcategory = df.groupby('Sub_Category', as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum'))
display(subcategory.sort_values('Profit'))


## 7. Regional analysis

In [ ]:
regional = df.groupby('Region', as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum'))
regional['Profit_Margin'] = regional['Profit']/regional['Sales']
display(regional.sort_values('Profit', ascending=False))


## 8. Discount vs profitability

In [ ]:
discount = df.groupby('Discount', as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum'))
discount['Profit_Margin'] = discount['Profit']/discount['Sales']
display(discount.sort_values('Discount'))

fig, ax = plt.subplots(figsize=(8,5))
sns.scatterplot(data=df, x='Discount', y='Profit', alpha=.35, ax=ax)
ax.set_title('Discount vs Transaction Profit')
plt.tight_layout(); plt.show()


## 9. Customer RFM segmentation

Reference date is one day after the latest order date. RFM scores are based on quintiles where possible.

In [ ]:
reference_date = df['Order_Date'].max() + pd.Timedelta(days=1)
rfm = df.groupby('Customer_ID').agg(
    Recency=('Order_Date', lambda x: (reference_date - x.max()).days),
    Frequency=('Order_ID','nunique'),
    Monetary=('Sales','sum')
).reset_index()

def qscore(s, reverse=False):
    # rank first prevents duplicate-value qcut failures
    ranked = s.rank(method='first')
    q = pd.qcut(ranked, 5, labels=[1,2,3,4,5]).astype(int)
    return 6-q if reverse else q

rfm['R_Score'] = qscore(rfm['Recency'], reverse=True)
rfm['F_Score'] = qscore(rfm['Frequency'])
rfm['M_Score'] = qscore(rfm['Monetary'])
rfm['RFM_Score'] = rfm[['R_Score','F_Score','M_Score']].sum(axis=1)

def segment(row):
    s = row['RFM_Score']
    if s >= 13: return 'Champions'
    if s >= 11: return 'Loyal Customers'
    if s >= 9: return 'Potential Loyalists'
    if s >= 7: return 'At Risk'
    return 'Lost / Low Value'

rfm['Segment'] = rfm.apply(segment, axis=1)
display(rfm.head())
display(rfm['Segment'].value_counts())


## 10. Customer segment economics

In [ ]:
customer_summary = rfm.groupby('Segment', as_index=False).agg(
    Customers=('Customer_ID','count'),
    Avg_Monetary=('Monetary','mean'),
    Total_Monetary=('Monetary','sum')
).sort_values('Total_Monetary', ascending=False)
display(customer_summary)


## 11. Top customers and loss-making products

In [ ]:
top_customers = df.groupby(['Customer_ID','Customer_Name'], as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum')).sort_values('Sales', ascending=False).head(10)
loss_products = df.groupby('Product_Name', as_index=False).agg(Sales=('Sales','sum'), Profit=('Profit','sum'))
loss_products = loss_products[loss_products['Profit'] < 0].sort_values('Profit').head(20)
display(top_customers)
display(loss_products)


## 12. Business recommendations

Write recommendations only after reviewing the calculated outputs above. Do not copy generic conclusions. Use the structure:

- **Finding:** what the data shows.
- **Impact:** why it matters financially/operationally.
- **Action:** what management should do.
- **Metric to monitor:** how success should be measured.

## 13. Export Power BI-ready tables

In [ ]:
out = Path('../outputs'); out.mkdir(exist_ok=True)
df.to_csv(out/'cleaned_superstore.csv', index=False)
rfm.to_csv(out/'customer_rfm.csv', index=False)
category.to_csv(out/'category_summary.csv', index=False)
regional.to_csv(out/'regional_summary.csv', index=False)
monthly.to_csv(out/'monthly_summary.csv', index=False)
print('Exported Power BI-ready files to', out.resolve())
